# Lesson 04: AI Agentic Design Principles

In this notebook, you will:
1. Build a Travel Agent grounded in **Transparency**, **Control**, and **Consistency**.
2. Implement runtime verbosity/tone customization and a `forget_user_data()` privacy reset.
3. Run an automated UX principle audit on agent responses.

In [ ]:
!pip install -q -U google-genai python-dotenv

In [ ]:
import os
from google import genai
from google.genai import types

def get_gemini_client():
    api_key = None
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
    api_key = api_key or os.environ.get("GEMINI_API_KEY")
    return genai.Client(api_key=api_key) if api_key else None

client = get_gemini_client()

class PrincipleTravelAgent:
    def __init__(self, verbosity="concise"):
        self.verbosity = verbosity
        self.history = []
        self.active = True

    def respond(self, query: str) -> str:
        if not self.active:
            return "Agent is paused by user control."
        self.history.append(query)
        if client:
            res = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=query,
                config=types.GenerateContentConfig(
                    system_instruction=f"You are a transparent AI Travel Assistant. Style: {self.verbosity}."
                )
            )
            return res.text
        return f"[Offline {self.verbosity} AI Response] Check Gate B12 35 mins before boarding."

    def forget(self):
        self.history.clear()
        return "All stored conversation history deleted."

agent = PrincipleTravelAgent(verbosity="concise")
print(agent.respond("What should I know before my flight?"))
print("History count before forget:", len(agent.history))
print(agent.forget(), "-> count:", len(agent.history))